# Gold Layer Tutorial: Silver → BERT Embeddings (Parquet)

This notebook walks through producing the **gold** layer by encoding silver reviews with BERT and writing Parquet.

**Data flow:**

1. Read JSONL from MinIO `silver/imdb/`
2. Encode text with BERT (`bert-base-uncased`)
3. Write model-ready Parquet to `gold/imdb/` (id, text, label, embedding)

**Prerequisites:**

- Silver data in MinIO (run `silver_layer_tutorial.ipynb` or `python -m src.transformation.silver_job`)
- MinIO running: `docker compose -f docker/docker-compose.yml up -d minio`
- Install: `pip install boto3 transformers torch pyarrow`

In [ ]:
!pip install -q boto3 transformers torch pyarrow

## 1. Gold schema

The gold layer is a **Parquet** table with:

| Field | Type | Description |
|-------|------|-------------|
| `id` | str | Unique identifier |
| `text` | str | Cleaned review text |
| `label` | int | 0 = negative, 1 = positive |
| `embedding` | list[float] | BERT pooler output (768 dims) |

## 2. Load silver from MinIO

Read silver JSONL files and collect records.

In [4]:
from src import config
from src.features.embedding_job import load_silver_records

silver_prefix = f"{config.SILVER_PREFIX}imdb/"
records = load_silver_records(silver_prefix)
print(f"Loaded {len(records)} silver records")

if not records:
    raise SystemExit("No silver data. Run silver_layer_tutorial or silver_job first.")

print(f"Sample: id={records[0]['id']}, label={records[0]['label']}, text[:60]={records[0]['text'][:60]}...")

Loaded 200 silver records
Sample: id=0, label=1, text[:60]=there is no relation at all between fortier and profiler but...


## 3. BERT encoding demo

Encode a few texts with BERT. The pooler output is a 768-dimensional vector per text.

In [5]:
import torch
from src.features.embedding_job import encode_batch
from transformers import AutoModel, AutoTokenizer

model_name = "bert-base-uncased"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

texts = [records[0]["text"][:200], records[1]["text"][:200]]
embeddings = encode_batch(model, tokenizer, texts, torch.device(device))
print(f"Input: {len(texts)} texts")
print(f"Output shape: {embeddings.shape}  (batch_size=2, hidden_size=768)")

Using device: cpu
Input: 2 texts
Output shape: torch.Size([2, 768])  (batch_size=2, hidden_size=768)


## 4. Run full embedding job (small sample)

Process a small sample and write gold Parquet. For production, use the CLI.

In [7]:
import io
import time

import pyarrow as pa
import pyarrow.parquet as pq

from src.utils.s3_client import ensure_bucket_exists, upload_bytes

# Use a small sample for the tutorial (e.g. 50 records)
sample = records
batch_size = 16

ids, texts_list, labels, embeddings_list = [], [], [], []
for i in range(0, len(sample), batch_size):
    batch = sample[i : i + batch_size]
    batch_texts = [r.get("text", "") for r in batch]
    enc = encode_batch(model, tokenizer, batch_texts, torch.device(device))
    for j, r in enumerate(batch):
        ids.append(str(r.get("id", "")))
        texts_list.append(r.get("text", ""))
        labels.append(int(r.get("label", 0)))
        embeddings_list.append(enc[j].cpu().numpy().tolist())

table = pa.table({
    "id": ids,
    "text": texts_list,
    "label": labels,
    "embedding": embeddings_list,
})

ensure_bucket_exists()
gold_prefix = f"{config.GOLD_PREFIX}imdb/"
key = f"{gold_prefix}imdb_gold_{int(time.time())}.parquet"
buf = io.BytesIO()
pq.write_table(table, buf, compression="snappy")
buf.seek(0)
upload_bytes(key, buf.read())
print(f"Wrote {len(sample)} records to s3://{config.S3_DATA_BUCKET}/{key}")

Wrote 200 records to s3://text-ml-data/gold/imdb/imdb_gold_1773713637.parquet


## 5. Verify gold Parquet

Download and read the Parquet file to inspect.

In [8]:
from src.utils.s3_client import list_objects, get_object_body

gold_objects = list_objects(gold_prefix)
print(f"Gold objects under {gold_prefix}: {len(gold_objects)}")
for obj in gold_objects[:5]:
    print(f"  - {obj['Key']}")

if gold_objects:
    pq_key = next(o["Key"] for o in gold_objects if o["Key"].endswith(".parquet"))
    body = get_object_body(pq_key)
    tbl = pq.read_table(io.BytesIO(body))
    print(f"\nTable: {tbl.num_rows} rows, {tbl.num_columns} columns")
    print(f"Columns: {tbl.column_names}")
    emb = tbl.column("embedding")[0]
    print(f"First embedding length: {len(emb)} (expected 768)")

Gold objects under gold/imdb/: 2
  - gold/imdb/imdb_gold_1773713555.parquet
  - gold/imdb/imdb_gold_1773713637.parquet

Table: 50 rows, 4 columns
Columns: ['id', 'text', 'label', 'embedding']
First embedding length: 768 (expected 768)


## 6. Run via CLI

To process all silver data at once:

```bash
python -m src.features.embedding_job
```

**Iceberg:** Write to an Iceberg table (ACID, time travel) instead of Parquet:

```bash
python -m src.features.embedding_job --iceberg
```

See `iceberg_gold_tutorial.ipynb` for why and how.

Other options:

```bash
python -m src.features.embedding_job --model bert-base-uncased --batch-size 32 --device cuda
```